библиотеки для агента

In [ ]:
!pip install -U langgraph langchain-openai
!pip install -U langchain-groq

In [ ]:
import json # объясняем почему не используем langchein
import os
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent

для ядра

In [ ]:
@tool
def Prediction_corolec(*args, **kwargs):
  """
  Эта функция не принимает аргументов. Она просто читает файл. Агент сам должен искать
  нужные данные из запроса человека.
  Этот инструмент нужен для того, чтобы можно было получить данные  с
  предсказаниями стоит ли покупать валюту, и если стоит то сколько. Также там даты
  """
  with open('ASFACK.json', 'r') as f:
    data = json.load(f)
  return data # прочитал данные предсказания



body

In [ ]:
llm = ChatGroq(model="Llama-3.1-8b-instant", api_key="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")
tools = [Prediction_corolec]
system_rules = """Ты — эксперт по крипто-трейдингу.
Твоя задача — анализировать данные из инструмента Prediction_corolec.
ПРАВИЛА ИНТЕРПРЕТАЦИИ:
1. Если значение LSTM (первое число) меньше 0.5 — это сигнал НЕ ПОКУПАТЬ (SELL/WAIT).
2. Если второе число 0.0 — это категорический запрет на покупку.
3. Никогда не выдумывай проценты (типа 13%), если их нет в данных.
Отвечай строго по цифрам из файла."""
agent = create_react_agent(llm, tools, prompt=system_rules)

/tmp/ipykernel_4727/2232105507.py:10: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=system_rules)


запуск

In [ ]:
def ask_agent(query):
  # 1. Формируем состояние: список сообщений с одним человеческим запросом
    initial_state = {"messages": [HumanMessage(content=query)]}

    # 2. Вызываем агента напрямую (без stream)
    # Это самый надежный способ для Groq
    response = agent.invoke(initial_state)

    # 3. Достаем последнее сообщение из результата
    # Обычно это ответ модели после вызова всех инструментов
    final_message = response["messages"][-1].content

    return final_message

try:
    print(ask_agent("Привет, посмотри данные в ядре и скажи, стоит ли мне покупать в эту 2026-05-10 21:00:00+00:00 по версии lstm?"))
except Exception as e:
    print(f"Опять ошибка, но теперь такая: {e}")

Покупать не стоит, так как первое число (LSTM) меньше 0.5.
